In [5]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


# Live — Stress Test

Push many frames at various resolutions to find the practical limits. Measures push latency, memory, and rendering responsiveness.

In [6]:
import time
import numpy as np
from quantem.widget import Live

## Benchmark: push latency by resolution

Measure time per push at 256, 512, 1K, 2K, and 4K. Each push sends one float32 frame over the widget comm channel.

In [7]:
sizes = [256, 512, 1024, 2048, 4096]
n_pushes = 10
print(f"{'Size':>6}  {'MB/push':>8}  {'ms/push':>8}  {'Total (s)':>10}")
print("-" * 40)
for sz in sizes:
    w = Live(buffer_size=n_pushes)
    t0 = time.perf_counter()
    for i in range(n_pushes):
        w.push(np.random.rand(sz, sz).astype(np.float32))
    elapsed = time.perf_counter() - t0
    mb = sz * sz * 4 / 1024 / 1024
    print(f"{sz:>6}  {mb:>7.1f}  {elapsed/n_pushes*1000:>7.0f}  {elapsed:>9.2f}")
    del w

  Size   MB/push   ms/push   Total (s)
----------------------------------------
   256      0.2        1       0.01
   512      1.0        2       0.02
  1024      4.0        8       0.08
  2048     16.0       65       0.65
  4096     64.0      179       1.79


## Scale test: 50 frames at 1K

Push 50 images and display the widget. Test thumbnail scrolling and navigation responsiveness.

In [8]:
w50 = Live(title="50 × 1K Stress", buffer_size=50)
t0 = time.perf_counter()
for i in range(50):
    w50.push(np.random.rand(1024, 1024).astype(np.float32), label=f"Frame {i}")
elapsed = time.perf_counter() - t0
print(f"50 × 1K pushed in {elapsed:.2f}s ({elapsed/50*1000:.0f} ms/push)")
w50

50 × 1K pushed in 0.80s (16 ms/push)


Live(50 frames, mode=2D)

## Scale test: 20 frames at 4K

Push 20 × 4096×4096 images (64 MB each). Tests the upper bound of the comm channel.

In [9]:
w4k = Live(title="20 × 4K Stress", buffer_size=20)
t0 = time.perf_counter()
for i in range(20):
    w4k.push(np.random.rand(4096, 4096).astype(np.float32), label=f"4K-{i}")
elapsed = time.perf_counter() - t0
print(f"20 × 4K pushed in {elapsed:.2f}s ({elapsed/20*1000:.0f} ms/push, {4096*4096*4/1024/1024:.0f} MB each)")
w4k

20 × 4K pushed in 5.12s (256 ms/push, 64 MB each)


Live(20 frames, mode=2D)

## Buffer overflow test

Push 100 frames with buffer_size=30 — oldest frames should be dropped from JS memory while count keeps increasing.

In [10]:
w_overflow = Live(title="Buffer Overflow (100 → 30)", buffer_size=30)
t0 = time.perf_counter()
for i in range(100):
    w_overflow.push(np.random.rand(512, 512).astype(np.float32), label=f"Frame {i}")
elapsed = time.perf_counter() - t0
print(f"100 × 512 pushed in {elapsed:.2f}s ({elapsed/100*1000:.0f} ms/push)")
print(f"n_frames (total pushed): {w_overflow.n_frames}")
print(f"buffer_size: {w_overflow.buffer_size}")
w_overflow

100 × 512 pushed in 0.33s (3 ms/push)
n_frames (total pushed): 100
buffer_size: 30


Live(100 frames, mode=2D)